In [3]:
import pandas as pd
import re
import string
from statistics import mean

df = pd.read_csv('news.csv')

# Take only 5000 samples
df = df.sample(n=5000, random_state=42).reset_index(drop=True)

print(f"Working with {len(df)} articles")

# Alternative subjectivity and polarity without textblob
# Using simple heuristics instead

def simple_subjectivity(text):
    """Estimate subjectivity based on opinion words"""
    opinion_words = ['think', 'believe', 'feel', 'should', 'would', 'could', 'maybe', 'perhaps', 'likely', 'probably']
    text_lower = text.lower()
    count = sum(1 for word in opinion_words if word in text_lower)
    return min(count / 10, 1.0)  # Cap at 1.0

def simple_polarity(text):
    """Estimate sentiment based on positive/negative word counts"""
    positive_words = ['good', 'great', 'excellent', 'amazing', 'wonderful', 'best', 'love', 'happy', 'perfect']
    negative_words = ['bad', 'terrible', 'awful', 'horrible', 'worst', 'hate', 'angry', 'dangerous', 'fear']
    
    text_lower = text.lower()
    pos_count = sum(1 for word in positive_words if word in text_lower)
    neg_count = sum(1 for word in negative_words if word in text_lower)
    
    total = pos_count + neg_count
    if total == 0:
        return 0
    return (pos_count - neg_count) / total

# Feature extraction function (without textblob)
def extract_features(text):
    text = str(text)
    
    features = {}
    
    # 1. Word count
    features['word_count'] = len(text.split())
    
    # 2. Exclamation count
    features['exclamation_count'] = text.count('!')
    
    # 3. Question count
    features['question_count'] = text.count('?')
    
    # 4. Capital letter ratio
    caps = sum(1 for c in text if c.isupper())
    features['capital_ratio'] = caps / len(text) if len(text) > 0 else 0
    
    # 5. Average word length
    words = text.split()
    if len(words) > 0:
        features['avg_word_length'] = mean([len(w) for w in words])
    else:
        features['avg_word_length'] = 0
    
    # 6. Simple subjectivity (no textblob)
    features['subjectivity'] = simple_subjectivity(text)
    
    # 7. Simple polarity (no textblob)
    features['polarity'] = simple_polarity(text)
    
    return features

# Extract features for all articles
print("Extracting features... (this may take 1-2 minutes)")
feature_list = []

for i, text in enumerate(df['text']):
    feature_list.append(extract_features(text))
    if i % 500 == 0:
        print(f"  Processed {i} articles...")

# Create feature DataFrame
X = pd.DataFrame(feature_list)
y = df['label'].map({'real': 1, 'fake': 0})

print(f"\n✅ Feature extraction complete!")
print(f"Features created: {list(X.columns)}")
print("\nFirst 5 rows of features:")
print(X.head())

# Now continue with model training
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib
import json

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Train model
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n🎯 MODEL ACCURACY: {accuracy * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Save model
joblib.dump(model, 'verisense_model.pkl')
print("\n✅ Model saved as 'verisense_model.pkl'")

# Save feature columns
with open('feature_columns.json', 'w') as f:
    json.dump(list(X.columns), f)
print("✅ Feature columns saved")

Working with 5000 articles
Extracting features... (this may take 1-2 minutes)
  Processed 0 articles...
  Processed 500 articles...
  Processed 1000 articles...
  Processed 1500 articles...
  Processed 2000 articles...
  Processed 2500 articles...
  Processed 3000 articles...
  Processed 3500 articles...
  Processed 4000 articles...
  Processed 4500 articles...

✅ Feature extraction complete!
Features created: ['word_count', 'exclamation_count', 'question_count', 'capital_ratio', 'avg_word_length', 'subjectivity', 'polarity']

First 5 rows of features:
   word_count  exclamation_count  question_count  capital_ratio  \
0         521                  0               0       0.025324   
1         549                  0               5       0.028982   
2         320                  0               0       0.053827   
3         305                  0               1       0.037313   
4         119                  0               0       0.062319   

   avg_word_length  subjectivity  pola